# CIFAR-100 Classification Training
Train a ResNet50 classifier on CIFAR-100 using PyTorch + timm + W&B.

**Flow:** Clone repo → Install deps → Set secrets → Run training

In [ ]:
# 1. Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 2. Clone repository
import os

REPO_URL = 'https://github.com/jetng1101/Training-model.git'
REPO_DIR = '/content/Training-model'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# dùng absolute path → không bị nested dù chạy cell nhiều lần
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 4. Mount Google Drive + Configure W&B
import os
from google.colab import drive, userdata

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100/checkpoints'
LOCAL_CHECKPOINT_DIR = 'checkpoints'

try:
    drive.mount('/content/drive', force_remount=True)
    CHECKPOINT_DIR = DRIVE_CHECKPOINT_DIR
    print(f'Drive mounted → checkpoints will be saved to Drive')
except Exception as e:
    print(f'Drive mount failed: {e}')
    print(f'Falling back to local checkpoints (will be lost on crash)')
    CHECKPOINT_DIR = LOCAL_CHECKPOINT_DIR

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
print(f'Checkpoint dir: {CHECKPOINT_DIR}')

In [ ]:
# 5. Run training (checkpoints saved to Drive, auto-resume on crash)
!python scripts/train.py \
    trainer.max_epochs=100 \
    trainer.checkpoint_dir={CHECKPOINT_DIR} \
    model.backbone.name=resnet50 \
    experiment_name=cifar100-resnet50-colab

In [ ]:
# 6. Verify checkpoint on Google Drive
import os

best_path = f'{CHECKPOINT_DIR}/best.pth'
if os.path.exists(best_path):
    size_mb = os.path.getsize(best_path) / 1024 / 1024
    print(f'best.pth found → {best_path} ({size_mb:.1f} MB)')
else:
    print('No best.pth yet — training may still be in progress')